# Downscaled `hurs` exploration

This notebook is for exploring downscaled relative humidity (hurs) data. It was designed to work with the zarr outputs from the 4km ERA5-based CMIP6 downscaling effort.

Because relative humidity is a bounded variable (0-100%), this notebook performs an out-of-range sanity check instead of extreme-high analysis. ECDF plots use Alaska fire season data (May-September).

In [ ]:
# Set BASE_DIR environment variable to your path
# export BASE_DIR=/path/to/cmip6_4km_downscaling

# Pass parameters via command line arguments like so, for example:
# papermill downscaled_hurs.ipynb output.ipynb -p models 'EC-Earth3-Veg' -p scenarios 'historical ssp370 ssp585'
models = "EC-Earth3-Veg"
scenarios = "historical ssp370 ssp585"

In [ ]:
# Parameters
models = "EC-Earth3-Veg"
scenarios = "historical ssp126 ssp370 ssp585"

In [ ]:
import math
import os
import numpy as np
import pandas as pd
import xarray as xr
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt
import gc
from baeda import projected_coords
from IPython.display import Markdown

import warnings
warnings.filterwarnings('ignore')

models = models.split(" ")
scenarios = scenarios.split(" ")

# Fire season months for ECDF plots
FIRE_SEASON_MONTHS = [5, 6, 7, 8, 9]

base_dir = Path(
    os.getenv("BASE_DIR", "/beegfs/CMIP6/crstephenson/EC-Earth3-Veg/cmip6_4km_downscaling")
)

ref_dir = base_dir.joinpath("era5_zarr")
cmip6_dir = base_dir.joinpath("cmip6_zarr")
adj_dir = base_dir.joinpath("adjusted")

random_points = []

In [ ]:
def open_hurs(model, scenario):
    zarr_store = adj_dir.joinpath(f"hurs_{model}_{scenario}_adjusted.zarr")
    ds = xr.open_zarr(zarr_store)
    hurs = ds.hurs  # units: %
    return hurs


def get_random_points(base, hurs_historical, n=5):
    if len(random_points) >= n:
        return random_points
    for _ in range(n):
        hurs_historical_extr = None
        while hurs_historical_extr is None or np.isnan(hurs_historical_extr).all():
            random_x = np.random.choice(base.x.values)
            random_y = np.random.choice(base.y.values)
            hurs_historical_extr = hurs_historical.sel(x=random_x, y=random_y)
        random_points.append((random_x, random_y))
    return random_points


def plot_out_of_range_hurs(model, scenario, hurs):
    """Plot maps and histograms showing out-of-range hurs values (< 0 or > 100%)."""
    hurs_mean = hurs.mean("time")

    hurs_oor = (hurs < 0) | (hurs > 100)
    hurs_oor_count = hurs_oor.sum("time")

    hurs_oor_values = hurs.where(hurs_oor).values.flatten()
    hurs_oor_values = hurs_oor_values[~np.isnan(hurs_oor_values)]

    fig, axs = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"Out-of-range hurs sanity check for {model}, {scenario}", fontsize=14)

    axs[0].imshow(
        hurs_mean.transpose("y", "x").values,
        cmap="Greys",
        alpha=0.5,
        interpolation="none",
    )

    masked_oor = np.ma.masked_where(
        hurs_oor_count.transpose("y", "x") == 0, hurs_oor_count.transpose("y", "x")
    )
    im = axs[0].imshow(masked_oor, cmap="Reds", alpha=0.8, interpolation="none")

    plt.colorbar(im, ax=axs[0], label="Count of out-of-range values (< 0 or > 100%)")
    axs[0].set_title("Mean hurs (grey) with out-of-range pixels (red overlay)")
    axs[0].set_xlabel("x")
    axs[0].set_ylabel("y")

    if hurs_oor_values.size > 0:
        axs[1].hist(hurs_oor_values, bins=30, color="red", alpha=0.7)
        axs[1].set_xlabel("hurs (%)")
        axs[1].set_ylabel("Frequency")
        axs[1].set_title("Histogram of out-of-range hurs values (< 0 or > 100%)")
        axs[1].text(
            0.98,
            0.98,
            f"{hurs_oor_values.size:,} out-of-range values\n({100 * hurs_oor_values.size / hurs.size:.4f}% of total)",
            ha="right",
            va="top",
            transform=axs[1].transAxes,
            fontsize=12,
            bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"),
        )
    else:
        axs[1].text(
            0.5,
            0.5,
            "No out-of-range hurs values found.",
            ha="center",
            va="center",
            fontsize=12,
        )
        axs[1].set_axis_off()

    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()

    del hurs
    gc.collect()


def count_out_of_range(ds_var):
    return ((ds_var < 0) | (ds_var > 100)).sum("time").load().sum().item()


def plot_fire_season_ecdfs(
    era5_ds, hist_ds, sim_ds, hurs_historical, hurs, pixels, names=None
):
    """Plot ECDFs filtered to fire season (May-September)."""
    n = len(pixels)
    ncols = math.ceil(n / 2)
    nrows = 2

    fig, axs = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4), sharey=True)
    axs = axs.flatten()

    model = sim_ds.attrs.get("source_id")
    scenario = sim_ds.attrs.get("experiment_id")

    fire_filter = lambda da: da.sel(time=da.time.dt.month.isin(FIRE_SEASON_MONTHS))

    for i, (x, y) in enumerate(pixels):
        era5_extr = era5_ds.rh2_mean.sel(x=x, y=y, method="nearest").rename("hurs")
        hist_extr = hist_ds.hurs.sel(x=x, y=y, method="nearest")
        sim_extr = sim_ds.hurs.sel(x=x, y=y, method="nearest")
        hurs_historical_extr = hurs_historical.sel(x=x, y=y, method="nearest").load()
        hurs_extr = hurs.sel(x=x, y=y, method="nearest").load()

        window_df = pd.concat(
            [
                fire_filter(era5_extr)
                .assign_coords(source="ERA5")
                .to_dataframe()
                .reset_index(),
                fire_filter(hist_extr)
                .rename("hurs")
                .assign_coords(source=f"{model}_historical")
                .to_dataframe()
                .reset_index(),
                fire_filter(sim_extr)
                .rename("hurs")
                .assign_coords(source=f"{model}_{scenario}")
                .to_dataframe()
                .reset_index(),
                fire_filter(hurs_historical_extr)
                .assign_coords(source=f"{model}_historical_downscaled")
                .to_dataframe()
                .reset_index(),
                fire_filter(hurs_extr)
                .rename("hurs")
                .assign_coords(source=f"{model}_{scenario}_downscaled")
                .to_dataframe()
                .reset_index(),
            ]
        )[["time", "source", "hurs"]]

        window_df.reset_index(inplace=True)

        color_mapping = {
            "ERA5": "black",
            f"{model}_historical": "lightblue",
            f"{model}_{scenario}": "lightgreen",
            f"{model}_historical_downscaled": "blue",
            f"{model}_{scenario}_downscaled": "green",
        }
        sns.ecdfplot(
            data=window_df,
            x="hurs",
            hue="source",
            ax=axs[i],
            palette=color_mapping,
        )
        axs[i].set_title(f"Pixel {i+1}\n(x={x:.0f}, y={y:.0f})" if names is None else names[i])
        axs[i].set_xlabel("hurs (%)")
        axs[i].set_ylabel("ECDF" if i % ncols == 0 else "")

    for j in range(i + 1, nrows * ncols):
        fig.delaxes(axs[j])

    label = "6 specified pixels" if names is not None else f"{n} pixels"
    plt.suptitle(
        f"ECDF of hurs for {label} for {model}.\nFire season data only (May-September)",
        fontsize=16,
    )
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


def plot_doy_means(era5_ds, hist_ds, sim_ds, hurs_historical, hurs, x, y):
    sim_doy_mean = sim_ds["hurs"].sel(x=x, y=y, method="nearest").groupby("time.dayofyear").mean(dim="time")
    hist_doy_mean = hist_ds["hurs"].sel(x=x, y=y, method="nearest").groupby("time.dayofyear").mean(dim="time")
    ref_doy_mean = era5_ds["rh2_mean"].sel(x=x, y=y, method="nearest").groupby("time.dayofyear").mean(dim="time")
    hurs_historical_doy_mean = hurs_historical.sel(x=x, y=y, method="nearest").groupby("time.dayofyear").mean(dim="time")
    hurs_doy_mean = hurs.sel(x=x, y=y, method="nearest").groupby("time.dayofyear").mean(dim="time")

    model = sim_ds.attrs.get("source_id")
    scenario = sim_ds.attrs.get("experiment_id")

    plt.figure(figsize=(9, 5))
    plt.plot(ref_doy_mean["dayofyear"], ref_doy_mean, label="ERA5", color="black")
    plt.plot(hist_doy_mean["dayofyear"], hist_doy_mean, label=f"{model} historical", color="lightblue")
    plt.plot(sim_doy_mean["dayofyear"], sim_doy_mean, label=f"{model} {scenario}", color="lightgreen")
    plt.plot(hurs_historical_doy_mean["dayofyear"], hurs_historical_doy_mean, label=f"{model} historical downscaled", color="blue")
    plt.plot(hurs_doy_mean["dayofyear"], hurs_doy_mean, label=f"{model} {scenario} downscaled", color="green")
    plt.xlabel("Day of Year")
    plt.ylabel("hurs (%)")
    plt.title(f"Day-of-year mean hurs: {model}, {scenario}, pixel: (x={x}, y={y})")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_random_ecdf(base, era5_ds, hist_ds, sim_ds, hurs_historical, hurs, ixy):
    """Plot fire season ECDF at a random point location."""
    model = sim_ds.attrs.get("source_id")
    scenario = sim_ds.attrs.get("experiment_id")

    fig, axs = plt.subplots(1, 2, figsize=(13, 6))
    fig.suptitle(
        f"hurs analysis for downscaled {model}, {scenario} at point location (fire season: May-Sep)",
        fontsize=14,
    )

    base.plot(ax=axs[0], cmap="Greys", alpha=0.5, add_colorbar=False, x="x")

    random_x = ixy[0]
    random_y = ixy[1]

    axs[0].scatter(random_x, random_y, color="blue", s=5, label="Random Pixel")

    fire_filter = lambda da: da.sel(time=da.time.dt.month.isin(FIRE_SEASON_MONTHS))

    era5_extr = era5_ds.rh2_mean.sel(x=random_x, y=random_y).rename("hurs")
    hist_extr = hist_ds.hurs.sel(x=random_x, y=random_y)
    sim_extr = sim_ds.hurs.sel(x=random_x, y=random_y)
    hurs_historical_extr = hurs_historical.sel(x=random_x, y=random_y).load()
    hurs_extr = hurs.sel(x=random_x, y=random_y).load()

    window_df = pd.concat(
        [
            fire_filter(era5_extr)
            .assign_coords(source="ERA5")
            .to_dataframe()
            .reset_index(),
            fire_filter(hist_extr)
            .rename("hurs")
            .assign_coords(source=f"{model}_historical")
            .to_dataframe()
            .reset_index(),
            fire_filter(sim_extr)
            .rename("hurs")
            .assign_coords(source=f"{model}_{scenario}")
            .to_dataframe()
            .reset_index(),
            fire_filter(hurs_historical_extr)
            .assign_coords(source=f"{model}_historical_downscaled")
            .to_dataframe()
            .reset_index(),
            fire_filter(hurs_extr)
            .rename("hurs")
            .assign_coords(source=f"{model}_{scenario}_downscaled")
            .to_dataframe()
            .reset_index(),
        ]
    )[["time", "source", "hurs"]]

    window_df.reset_index(inplace=True)

    color_mapping = {
        "ERA5": "black",
        f"{model}_historical": "lightblue",
        f"{model}_{scenario}": "lightgreen",
        f"{model}_historical_downscaled": "blue",
        f"{model}_{scenario}_downscaled": "green",
    }

    sns.ecdfplot(
        data=window_df,
        x="hurs",
        hue="source",
        ax=axs[1],
        palette=color_mapping,
    )
    axs[1].set_xlabel("hurs (%)")

    plt.tight_layout()
    plt.show()

    return window_df

In [ ]:
era5_ds = xr.open_dataset(ref_dir.joinpath("rh2_mean_era5.zarr"))

### Choose hand-picked places

In [ ]:
places1 = ["Fairbanks", "Anchorage", "Nome", "Yakutat", "Utqiagvik", "near_McGrath"]
pixels1 = [(projected_coords[k]["x"], projected_coords[k]["y"]) for k in places1]

places2 = [
    "near_McGrath",
    "near_Arctic_Village",
    "warmest",
    "coldest",
    "wettest",
    "driest",
]
pixels2 = [(projected_coords[k]["x"], projected_coords[k]["y"]) for k in places2]

In [ ]:
for model in models:
    display(Markdown(f"# {model}"))
    hist_ds = xr.open_dataset(cmip6_dir.joinpath(f"hurs_{model}_historical.zarr"))
    hurs_historical = open_hurs(model, "historical")

    era5_oor_count = count_out_of_range(era5_ds.rh2_mean)
    historical_oor_count = count_out_of_range(hist_ds.hurs)
    adj_historical_oor_count = count_out_of_range(hurs_historical)

    for scenario in scenarios:
        display(Markdown(f"## {model}, {scenario}"))
        hurs = open_hurs(model, scenario)
        sim_ds = xr.open_dataset(cmip6_dir.joinpath(f"hurs_{model}_{scenario}.zarr"))

        sim_oor_count = count_out_of_range(sim_ds.hurs)
        adj_oor_count = count_out_of_range(hurs)

        display(Markdown(f"### Out-of-range sanity check (valid range: 0-100%)"))
        plot_out_of_range_hurs(model, scenario, hurs)

        print("Counts of out-of-range hurs values (< 0 or > 100%) across all days:")
        print(f"ERA5: {era5_oor_count:,} pixels")
        if scenario != "historical":
            print(f"{model} historical (unadjusted): {historical_oor_count:,} pixels")
            print(f"{model} historical (downscaled): {adj_historical_oor_count:,} pixels")
        print(f"{model} {scenario} (unadjusted): {sim_oor_count:,} pixels")
        print(f"{model} {scenario} (downscaled): {adj_oor_count:,} pixels")

        if scenario != "historical":
            display(Markdown(f"### Hand-picked places (fire season: May-Sep)"))
            plot_fire_season_ecdfs(era5_ds, hist_ds, sim_ds, hurs_historical, hurs, pixels1, names=places1)
            plot_fire_season_ecdfs(era5_ds, hist_ds, sim_ds, hurs_historical, hurs, pixels2, names=places2)

            display(Markdown(f"### Day-of-year (DOY) means"))
            for place, (x, y) in zip(places1, pixels1):
                plot_doy_means(era5_ds, hist_ds, sim_ds, hurs_historical, hurs, x=x, y=y)

            display(Markdown(f"### Random locations (fire season: May-Sep)"))
            base = hurs.mean("time")
            random_points = get_random_points(base, hurs_historical)
            for ixy in random_points:
                plot_random_ecdf(base, era5_ds, hist_ds, sim_ds, hurs_historical, hurs, ixy)